# Checkpoint 3 — Churn model EDA + train + SHAP

Loads `telco_churn.features` from BigQuery, trains XGBoost for `target_churn_90d`,
evaluates holdout metrics, and inspects global / local SHAP for migrated users.

For a non-notebook train path: `python models/train_churn_model.py`

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import shap
from google.cloud import bigquery
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import train_test_split
import xgboost as xgb

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from models.preprocess import build_feature_matrix, load_feature_config
from models.churn_model_service import predict_risk_and_drivers, get_global_shap_summary

PROJECT_ID = os.environ.get("PROJECT_ID", "project-5506c1fb-580d-4ca3-b04")
DATASET = os.environ.get("BIGQUERY_DATASET", "telco_churn")
print("ROOT", ROOT)
print("PROJECT_ID", PROJECT_ID)

In [ ]:
client = bigquery.Client(project=PROJECT_ID)
df = client.query(f"SELECT * FROM `{PROJECT_ID}.{DATASET}.features`").to_dataframe()
print(df.shape)
display(df[["customer_id", "migration_flag", "post_migration_qos", "bill_change_pct",
            "usage_change_30d", "support_sentiment", "value_segment", "target_churn_90d"]].head())
print(df.groupby("migration_flag")["target_churn_90d"].mean())

In [ ]:
config = load_feature_config(ROOT / "models" / "feature_config.json")
X = build_feature_matrix(df, config)
y = df[config["target_column"]].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
pos, neg = int(y_train.sum()), int(len(y_train) - y_train.sum())
model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.08,
    subsample=0.9, colsample_bytree=0.9, objective="binary:logistic",
    eval_metric="auc", scale_pos_weight=neg / max(pos, 1), random_state=42, n_jobs=4,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print("ROC-AUC", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred, digits=3))
RocCurveDisplay.from_predictions(y_test, proba)
plt.title("Holdout ROC — target_churn_90d")
plt.show()

In [ ]:
# Use XGBoost pred_contribs (TreeExplainer breaks on XGBoost 3 base_score encoding).
from models.explain import shap_values_for_matrix

sample = X_test.sample(n=min(400, len(X_test)), random_state=42)
sv = shap_values_for_matrix(model, sample)
shap.summary_plot(sv, sample, max_display=15, show=False)
plt.title("Global SHAP (pred_contribs) — holdout sample")
plt.tight_layout()
plt.show()


In [ ]:
# Local explanations for sample migrated users (via reusable service after train script).
# Prefer artifacts from: python models/train_churn_model.py
migrants = df[df["migration_flag"] == 1].head(5)
for _, row in migrants.iterrows():
    result = predict_risk_and_drivers(row)
    print(result["customer_id"], "risk=", round(result["risk"], 3),
          "top_cat=", result["top_driver_category"])
    for d in result["drivers"][:3]:
        print(" ", d)